# Evidence-gated DFTpy–QE divacancy comparison

Pair distance alone cannot identify equivalent calculations. This notebook requires qualified cases, the same verified starting geometry and direction, atom counts, XC, pressure, relaxation and energy/reference definitions. QE basis convergence and pseudopotential validation need independent evidence. The DFTpy grid must never be relabeled as a QE plane-wave cutoff.

No QE divacancy result is fabricated or substituted from monovacancy data. Supply an independently audited QE CSV via `AL_DEFECTS_QE_DIVACANCY_AUDIT`; absent or incomplete evidence produces a diagnostic message and no comparison curve.


In [ ]:
from pathlib import Path
import os
import sys

candidates = ([Path(os.environ['AL_DEFECTS_REPO'])] if os.environ.get('AL_DEFECTS_REPO') else [])
candidates += [Path.cwd(), *Path.cwd().parents]
REPO = next((p.resolve() for p in candidates if (p / 'scripts' / 'divacancy_analysis_checks.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the repository, or set AL_DEFECTS_REPO to the full repository')
sys.path.insert(0, str(REPO / 'scripts'))
from divacancy_analysis_checks import latest_dftpy_root
import csv
import pandas as pd
import matplotlib.pyplot as plt
from collect_dftpy_conventional_vacancy import collect_scan
from divacancy_analysis_checks import compatible_comparison
DFTPY_ROOT = latest_dftpy_root(REPO)
qe_path = Path(os.environ['AL_DEFECTS_QE_DIVACANCY_AUDIT']).expanduser() if os.environ.get('AL_DEFECTS_QE_DIVACANCY_AUDIT') else None
qe_rows = []
if qe_path is not None:
    if not qe_path.is_file():
        raise FileNotFoundError(f'QE audit CSV not found: {qe_path}')
    with qe_path.open(encoding='utf-8-sig', newline='') as handle:
        qe_rows = list(csv.DictReader(handle))


In [ ]:
dftpy_rows = collect_scan(DFTPY_ROOT, 'pair_scan')
comparison_rows, diagnostics = compatible_comparison(dftpy_rows, qe_rows)
for diagnostic in diagnostics:
    print(diagnostic)
comparison = pd.DataFrame(comparison_rows)
if comparison.empty:
    print('No compatible independently qualified DFTpy–QE pairs; quantitative comparison unavailable.')
else:
    comparison = comparison.sort_values(['direction', 'r_A'])
    display(comparison)


In [ ]:
if not comparison.empty:
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    for direction, group in comparison.groupby('direction'):
        ax.plot(group.r_A, group.DFTpy_E_2vac_eV.astype(float), 'o-', label=f'DFTpy {direction}')
        ax.plot(group.r_A, group.QE_E_2vac_eV.astype(float), 's-', label=f'QE {direction}')
    ax.set(xlabel='Initial minimum-image vacancy distance (Å)', ylabel='Total two-vacancy formation energy (eV)')
    ax.legend(frameon=False)
    fig.tight_layout()
